# Sevastopol AI — бот (одна ячейка)

1. Нажми ▶️ — ячейка сама спросит токен **@BotFather** (ввод через `getpass`
   не сохраняется в файле ноутбука), склонирует репозиторий, поставит зависимости
   и запустит бота.
2. Токен можно не вводить каждый раз: Colab → 🔑 **Secrets** → добавь `TG_TOKEN`
   (и, если нужно, `ADMIN_ID`) → разреши доступ ноутбуку. Ячейка возьмёт их сама.
3. Остановка — ■ (прервать ячейку). Colab засыпает без активности, поэтому для
   24/7 нужен сервер (README → «Запуск 24/7»).

⚠️ Никогда не вписывай токен строкой в ячейку: он уедет в git вместе с ноутбуком.
Засветился — перевыпусти: @BotFather → /mybots → API Token → **Revoke**
(подробности — `SECURITY.md`).


In [ ]:
# ═══════════ Sevastopol AI — запуск бота одной ячейкой ═══════════
ADMIN_ID = ""     # твой Telegram ID (узнать: @userinfobot) — для /admin и заявок с формы

import os, shutil, subprocess, sys

def colab_secret(name):
    try:
        from google.colab import userdata
        return str(userdata.get(name) or "").strip()
    except Exception:
        return ""

TG_TOKEN = colab_secret("TG_TOKEN")
if not TG_TOKEN:
    import getpass
    TG_TOKEN = getpass.getpass("TG_TOKEN от @BotFather: ").strip()
if not ADMIN_ID:
    ADMIN_ID = colab_secret("ADMIN_ID")

REPO = "https://github.com/Yurich-citycode/SevastopolAIbot.git"
DIR = "/content/SevastopolAIbot"


def sh(*cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


if os.path.isdir(DIR):
    if subprocess.run(["git", "-C", DIR, "pull", "--ff-only", "origin", "main"]).returncode:
        shutil.rmtree(DIR)
if not os.path.isdir(DIR):
    sh("git", "clone", "--depth", "1", REPO, DIR)

sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", cwd=DIR)

# настройки передаём процессу окружением — файл .env на диске не создаём
settings = dict(os.environ)
settings.update({
    "TG_TOKEN": TG_TOKEN,
    "ADMIN_ID": ADMIN_ID,
    "SPREADSHEET_ID": "1RaHoS_8Ov-kNKSZJK015ceC6H3fsWnW-D-8Yee4ckQI",
    "NEWS_CHANNEL_URL": "https://t.me/Sevastopol_AI",
    "REFRESH_SECONDS": "600",
    "STATE_FILE": "bot_state.json",
})

p = subprocess.Popen([sys.executable, "-u", "sevastopolaibot.py"], cwd=DIR, env=settings,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
print("\n=== бот завершился с кодом", p.wait(), "===")
